In [1]:
import pandas as pd
import sqlite3
import os

project_path = os.path.join(os.path.expanduser("~"), "Documents", "Healthcare_SQL_Analytics")
db_path = os.path.join(project_path, "healthcare_readmissions.db")

conn = sqlite3.connect(db_path)

def run(sql):
    return pd.read_sql(sql, conn)

run("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")

,name
0,bridge_encounter_diagnosis
1,dim_age
2,dim_diagnosis
3,dim_specialty
4,fact_encounters


In [10]:
run("""
SELECT
    CASE
        WHEN n_emergency = 0 THEN '0 ER visits'
        WHEN n_emergency = 1 THEN '1 ER visit'
        ELSE '2+ ER visits'
    END AS prior_er_group,
    COUNT(*) AS encounters,
    ROUND(AVG(CASE WHEN readmitted = 'yes' THEN 1.0 ELSE 0 END) * 100, 1) AS readmit_pct
FROM fact_encounters
GROUP BY prior_er_group
ORDER BY MIN(n_emergency);
""")

,prior_er_group,encounters,readmit_pct
0,0 ER visits,22272,45.1
1,1 ER visit,1842,59.0
2,2+ ER visits,886,71.1


In [2]:
run("""
SELECT
    CASE
        WHEN n_inpatient = 0 THEN '0 prior stays'
        WHEN n_inpatient = 1 THEN '1 prior stay'
        WHEN n_inpatient BETWEEN 2 AND 3 THEN '2-3 prior stays'
        ELSE '4+ prior stays'
    END AS prior_inpatient_group,
    COUNT(*) AS encounters,
    ROUND(AVG(CASE WHEN readmitted = 'yes' THEN 1.0 ELSE 0 END) * 100, 1) AS readmit_pct
FROM fact_encounters
GROUP BY prior_inpatient_group
ORDER BY MIN(n_inpatient);
""")

,prior_inpatient_group,encounters,readmit_pct
0,0 prior stays,16537,39.9
1,1 prior stay,4926,54.7
2,2-3 prior stays,2742,66.4
3,4+ prior stays,795,81.3


In [3]:
run("""
SELECT
    a1c_test,
    COUNT(*) AS encounters,
    ROUND(AVG(CASE WHEN readmitted = 'yes' THEN 1.0 ELSE 0 END) * 100, 1) AS readmit_pct
FROM fact_encounters
GROUP BY a1c_test
ORDER BY readmit_pct DESC;
""")

,a1c_test,encounters,readmit_pct
0,no,20938,47.4
1,high,2827,45.9
2,normal,1235,42.1


In [9]:
run("""
SELECT
    diabetes_med,
    COUNT(*) AS encounters,
    ROUND(AVG(CASE WHEN readmitted = 'yes' THEN 1.0 ELSE 0 END) * 100, 1) AS readmit_pct
FROM fact_encounters
GROUP BY diabetes_med
ORDER BY readmit_pct DESC;
""")

,diabetes_med,encounters,readmit_pct
0,yes,19228,48.7
1,no,5772,41.4


In [8]:
run("""
SELECT
    glucose_test,
    COUNT(*) AS encounters,
    ROUND(AVG(CASE WHEN readmitted = 'yes' THEN 1.0 ELSE 0 END) * 100, 1) AS readmit_pct
FROM fact_encounters
GROUP BY glucose_test
ORDER BY readmit_pct DESC;
""")

,glucose_test,encounters,readmit_pct
0,high,686,52.0
1,normal,689,48.3
2,no,23625,46.8


In [6]:
run("""
SELECT
    d.diagnosis_category AS primary_diagnosis,
    COUNT(*) AS encounters,
    ROUND(AVG(CASE WHEN f.readmitted = 'yes' THEN 1.0 ELSE 0 END) * 100, 1) AS readmit_pct
FROM fact_encounters f
JOIN bridge_encounter_diagnosis b
    ON f.encounter_id = b.encounter_id
JOIN dim_diagnosis d
    ON b.diagnosis_id = d.diagnosis_id
WHERE b.diagnosis_position = 1
GROUP BY d.diagnosis_category
ORDER BY readmit_pct DESC;
""")

,primary_diagnosis,encounters,readmit_pct
0,Diabetes,1747,53.6
1,Missing,4,50.0
2,Respiratory,3680,49.1
3,Circulatory,7824,47.9
4,Digestive,2329,47.4
5,Other,6498,45.1
6,Injury,1666,43.6
7,Musculoskeletal,1252,39.5


In [7]:
run("""
SELECT
    s.specialty_name,
    COUNT(*) AS encounters,
    ROUND(AVG(CASE WHEN f.readmitted = 'yes' THEN 1.0 ELSE 0 END) * 100, 1) AS readmit_pct
FROM fact_encounters f
JOIN dim_specialty s
    ON f.specialty_id = s.specialty_id
GROUP BY s.specialty_name
ORDER BY readmit_pct DESC;
""")

,specialty_name,encounters,readmit_pct
0,Family/GeneralPractice,1882,49.5
1,Emergency/Trauma,1885,49.4
2,Missing,12382,48.9
3,Cardiology,1409,45.0
4,InternalMedicine,3565,44.8
5,Other,2664,41.5
6,Surgery,1213,41.2
